# YaRN — Yet another RoPE eNtension

源码导航：[core/position/yarn.py](../../../core/position/yarn.py) 中的 `YarnRotaryPositionalEmbedding`。

Peng et al. (2023) 在 *YaRN: Efficient Context Window Extension of Large Language Models* 中提出 YaRN，通过**频率感知插值**与**注意力温度缩放**两大机制，在不进行任何微调的情况下将预训练模型的上下文窗口扩展 2×–8×。YaRN 是目前长上下文外推的主流方案之一，已被 LLaMA 2 Long、Mistral 等模型采用。

### 1. 理论推导

#### 1.1 问题背景：RoPE 的外推困境

标准 RoPE 的 cos/sin 表基于训练时的最大长度 $T_{\text{train}}$ 预计算。当推理序列长度 $T_{\text{test}} > T_{\text{train}}$ 时，Q/K 会旋转到训练时从未见过的角度，导致注意力分布崩溃（ppl 激增）。

#### 1.2 方案一：NTK-aware 频率插值（by-parts）

朴素的线性插值（Position Interpolation, PI）将所有频率统一除以扩展因子 $s$：

$$\theta'_i = \theta_i / s$$

但这会过度压缩高频分量（小波长），导致局部精细位置信息丢失。

YaRN 的改进：根据波长在 $T_{\text{train}}$ 上的覆盖程度，对不同维度采用不同的插值策略：
- **低频维度**（波长远大于 $T_{\text{train}}$）：几乎不插值，保持全局结构；
- **高频维度**（波长远小于 $T_{\text{train}}$）：完全插值，因为它们在训练长度内已完成多个周期；
- **中间维度**：线性过渡。

实现上通过 `_yarn_find_correction_range` 计算插值过渡区间 $[\text{low}, \text{high}]$，并构造 ramp 函数：

$$\text{ramp}(i) = \text{clamp}\left(\frac{i - \text{low}}{\text{high} - \text{low}}, 0, 1\right)$$

最终频率缩放因子：

$$\text{scale}(i) = \frac{1}{s} \cdot (1 - \text{ramp}(i)) + \text{ramp}(i)$$

#### 1.3 方案二：注意力温度缩放

长上下文使注意力 logits 的分布更平坦，softmax 趋向均匀分布，导致信息稀释。YaRN 引入温度缩放：

$$\text{score}'(i, j) = \frac{\text{score}(i, j)}{\sqrt{t}}, \quad t \approx 0.1 \cdot \log(s) + 1.0$$

其中 $t$ 随扩展因子 $s$ 缓慢增长，抵消长距离注意力分散效应。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.position.yarn import YarnRotaryPositionalEmbedding, _yarn_find_correction_range
from core.position.rope import RotaryPositionalEmbedding, apply_rope

### 2. 插值范围计算验证

In [ ]:
import math

half = 64  # head_dim=128 时的 half
low, high = _yarn_find_correction_range(
    low_rot=32, high_rot=1,
    d_model=half, base=1e6, max_position_embeddings=2048
)
print(f"过渡区间: [{low}, {high}]")
print(f"低频端（i < {low}）: 几乎不插值")
print(f"高频端（i > {high}）: 完全插值（scale = 1/s）")
assert low >= 0 and high < half, "区间应在有效范围内"

### 3. 形状与接口兼容性

`YarnRotaryPositionalEmbedding` 的 `forward` 返回 `(cos, sin)`，与标准 `RotaryPositionalEmbedding` 完全兼容，可直接替换。

In [ ]:
head_dim = 64
yarn = YarnRotaryPositionalEmbedding(
    head_dim=head_dim,
    max_seq_len=4096,
    base=1e6,
    scale_factor=4.0,           # 扩展 4 倍
    orig_max_seq_len=2048,      # 原始训练长度
)

cos, sin = yarn(seq_len=512)
print("cos.shape:", tuple(cos.shape))   # (512, 64)
print("sin.shape:", tuple(sin.shape))   # (512, 64)
assert cos.shape == (512, head_dim), "缓存形状应匹配"

### 4. 频率缩放对比：RoPE vs YaRN

对比相同 `base=1e6` 下，标准 RoPE 与 YaRN (scale=4.0) 的 inv_freq 分布差异。

In [ ]:
import matplotlib.pyplot as plt

rope = RotaryPositionalEmbedding(head_dim=64, max_seq_len=128, base=1e6)
yarn = YarnRotaryPositionalEmbedding(head_dim=64, max_seq_len=128, base=1e6,
                                     scale_factor=4.0, orig_max_seq_len=2048)

rope_freq = rope.inv_freq.numpy()
yarn_freq = yarn.inv_freq.numpy()
dims = torch.arange(len(rope_freq)).numpy()

plt.figure(figsize=(10, 4))
plt.plot(dims, rope_freq, label="RoPE (no scaling)", marker="o", markersize=3)
plt.plot(dims, yarn_freq, label="YaRN (scale=4.0)", marker="s", markersize=3)
plt.xlabel("Dimension index i")
plt.ylabel("Inverse frequency")
plt.title("RoPE vs YaRN Inverse Frequency Distribution")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("RoPE 低频端 freq:", rope_freq[0])
print("YaRN 低频端 freq:", yarn_freq[0])
print("RoPE 高频端 freq:", rope_freq[-1])
print("YaRN 高频端 freq:", yarn_freq[-1])

### 5. 注意力温度缩放验证

In [ ]:
import math

for s in [1.0, 2.0, 4.0, 8.0, 16.0]:
    yarn = YarnRotaryPositionalEmbedding(
        head_dim=64, max_seq_len=128, base=1e6,
        scale_factor=s, orig_max_seq_len=2048
    )
    expected = 0.1 * math.log(s) + 1.0 if s > 1.0 else 1.0
    print(f"scale_factor={s:5.1f}:  attn_factor={yarn.attn_factor:.4f}  (expected={expected:.4f})")
    assert abs(yarn.attn_factor - expected) < 1e-6, "温度因子应符合预期公式"

### 6. 源码精讲

```python
class YarnRotaryPositionalEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len=8192, base=1e6,
                 scale_factor=1.0, orig_max_seq_len=2048,
                 beta_fast=32, beta_slow=1, attn_factor=None):
        super().__init__()
        # ... 初始化参数 ...
        self._build_scaled_freqs()
        self._build_cache(max_seq_len, device=..., dtype=torch.float32)

    def _build_scaled_freqs(self):
        freqs = 1.0 / (base ** (torch.arange(0, half) * 2.0 / head_dim))
        if scale_factor <= 1.0:
            self.register_buffer("inv_freq", freqs)
            return
        # 找到插值过渡区间 [low, high]
        low, high = _yarn_find_correction_range(...)
        # 构造 ramp：0 -> 1 的线性过渡
        ramp = torch.zeros(half)
        ramp[low:high+1] = torch.linspace(0.0, 1.0, high-low+1)
        # 频率缩放：低频保持，高频压缩
        freq_scale = (1.0 / scale_factor) * (1.0 - ramp) + ramp
        self.register_buffer("inv_freq", freqs * freq_scale)
```

关键设计点：
- `_yarn_find_correction_range` 通过波长反推维度索引，确定哪些维度应被插值。
- `ramp` 函数实现了从"不插值"到"完全插值"的平滑过渡。
- `apply_attention_scaling` 方法在 attention logits 上施加温度缩放，使用时需显式调用。

---

## 延伸阅读与参考资料

### 核心论文
- **YaRN: Efficient Context Window Extension of Large Language Models**: Peng et al., 2023. [arXiv:2309.00071](https://arxiv.org/abs/2309.00071)

### 相关工作
- **NTK-Aware Scaling**: 无微调扩展上下文窗口的早期方案，YaRN 在此基础上改进。
- **Position Interpolation (PI)**: Chen et al., 2023. 线性插值所有频率的朴素方案。
- **LLaMA 2 Long**: 采用 YaRN 实现 32K 上下文扩展。